In [ ]:
# ────────────────────────────────────────────────
# copy the dataset to the working directory
# ────────────────────────────────────────────────

import shutil
import os
source_base = "/kaggle/input/datasets/moatazbillahgahgah/ar-jp-dataset/"     
dest_base = "/kaggle/working/"




if os.path.exists(source_base):
    shutil.copytree(source_base,dest_base, dirs_exist_ok=True)
    print("Copied")
else:
    print("Folder")

print("Done! Check:")
!ls -l /kaggle/working/

In [ ]:
# ────────────────────────────────────────────────
# filter the dataset
# ────────────────────────────────────────────────

import re
from pathlib import Path
from langdetect import detect, LangDetectException
from tqdm import tqdm

def clean_text(text: str) -> str:
    """Remove control chars, normalize spaces, strip"""
    text = re.sub(r'[\x00-\x1F\x7F-\x9F]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def is_reasonable_length_ratio(ja: str, ar: str, min_ratio=0.33, max_ratio=3.0):
    ja_len = len(ja.split())
    ar_len = len(ar.split())
    if ja_len == 0 or ar_len == 0:
        return False
    ratio = ja_len / ar_len
    return min_ratio <= ratio <= max_ratio

def is_too_short(ja: str, ar: str, min_words=3):
    return len(ja.split()) < min_words or len(ar.split()) < min_words

def has_too_much_punctuation(text: str, max_ratio=0.45):
    punct = sum(1 for c in text if c in r'.,!?…「」『』・ー―〜☆★♡♥')
    return (punct / max(1, len(text))) > max_ratio

def is_likely_correct_language(text: str, expected_lang: str, min_confidence=0.7):
    try:
        lang = detect(text)
        return lang == expected_lang
    except LangDetectException:
        return False  # if detection fails, skip or accept

def filter_pair(ja: str, ar: str) -> bool:
    ja_clean = clean_text(ja)
    ar_clean = clean_text(ar)

    if not ja_clean or not ar_clean:
        return False

    # Basic length checks
    if is_too_short(ja_clean, ar_clean):
        return False

    if not is_reasonable_length_ratio(ja_clean, ar_clean):
        return False

    if has_too_much_punctuation(ja_clean) or has_too_much_punctuation(ar_clean):
        return False

    # Optional: language check (uncomment if you want stricter filtering)
    # if not is_likely_correct_language(ja_clean, 'ja') or not is_likely_correct_language(ar_clean, 'ar'):
    #     return False

    return True

def filter_parallel_files(
    src_path: str,
    trg_path: str,
    out_src: str = "ja_filtered.txt",
    out_trg: str = "ar_filtered.txt"
):
    src_path = Path(src_path)
    trg_path = Path(trg_path)

    if not src_path.is_file() or not trg_path.is_file():
        print(f"Error: One or both files missing:\n  {src_path}\n  {trg_path}")
        return

    kept = 0
    total = 0

    with (
        src_path.open(encoding="utf-8") as f_src,
        trg_path.open(encoding="utf-8") as f_trg,
        open(out_src, "w", encoding="utf-8") as f_out_src,
        open(out_trg, "w", encoding="utf-8") as f_out_trg
    ):
        # Use tqdm for progress bar
        lines = zip(f_src, f_trg)
        for ja_line, ar_line in tqdm(lines, desc="Filtering pairs", unit="pair"):
            total += 1
            ja = ja_line.strip()
            ar = ar_line.strip()

            if filter_pair(ja, ar):
                kept += 1
                print(ja, file=f_out_src)
                print(ar, file=f_out_trg)

            if total % 100_000 == 0:
                print(f"Processed {total:,} | Kept {kept:,} ({kept/total:.1%})")

    print(f"\nFinished filtering:")
    print(f"Total pairs:     {total:,}")
    print(f"Kept pairs:      {kept:,} ({kept/total:.1%} kept)")
    print(f"Output files:")
    print(f"  → {out_src}")
    print(f"  → {out_trg}")

# ────────────────────────────────────────────────
# Run it — adjust paths if your filenames are different

if __name__ == "__main__":
    filter_parallel_files(
        src_path = "/kaggle/working/OpenSubtitles.ar-ja.ja",   # Japanese
        trg_path = "/kaggle/working/OpenSubtitles.ar-ja.ar",   # Arabic
        out_src  = "/kaggle/working/ja_filtered.txt",
        out_trg  = "/kaggle/working/ar_filtered.txt"
    )

In [ ]:
!head -n 10 /kaggle/working/ja_filtered.txt
!head -n 10 /kaggle/working/ar_filtered.txt

In [144]:
# ==============================
# 3. Load your parallel dataset
# ==============================

src_file = "/kaggle/working/ja_filtered.txt"
tgt_file = "/kaggle/working/ar_filtered.txt"

ja_lines = []
ar_lines = []

with open(src_file, encoding="utf-8") as f_src, open(tgt_file, encoding="utf-8") as f_trg:

    for ja_line, ar_line in tqdm(zip(f_src, f_trg)):
        ja_line = ja_line.strip()
        ar_line = ar_line.strip()

        if ja_line and ar_line:
            ja_lines.append(ja_line)
            ar_lines.append(ar_line)

assert len(ja_lines) == len(ar_lines)

print("Total sentence pairs:", len(ja_lines))






Checking files...
-rw-r--r-- 1 root root 11M Mar 30 15:46 /kaggle/working/ar_filtered.txt
-rw-r--r-- 1 root root 10M Mar 30 15:46 /kaggle/working/ja_filtered.txt
ja_filtered.txt  → 166,195 lines
ar_filtered.txt  → 166,195 lines
Line counts match → good to proceed



Generating ja split: 0 examples [00:00, ? examples/s]

Generating ar split: 0 examples [00:00, ? examples/s]

Raw dataset structure:
DatasetDict({
    ja: Dataset({
        features: ['text'],
        num_rows: 166195
    })
    ar: Dataset({
        features: ['text'],
        num_rows: 166195
    })
})

Combined parallel dataset:
Dataset({
    features: ['ja', 'ar'],
    num_rows: 166195
})
First example:
{'ja': 'でも\u3000選ぶのは彼女。 どうなっても\u3000友達でいよう', 'ar': 'سنترك الإختيار لها ولكن مهما كان من إختارته سنبقى نحن أصدقاء'}

Final DatasetDict with splits:
DatasetDict({
    train: Dataset({
        features: ['ja', 'ar'],
        num_rows: 12465
    })
    validation: Dataset({
        features: ['ja', 'ar'],
        num_rows: 1661
    })
    test: Dataset({
        features: ['ja', 'ar'],
        num_rows: 2493
    })
})
Train examples:      12,465
Validation examples: 1,661
Test examples:       2,493


In [127]:
# ==============================
# 4. Create dataset
# ==============================

dataset = Dataset.from_dict({
    "ja": ja_lines,
    "ar": ar_lines
})

In [128]:
# ==============================
# 5. Train / validation split
# ==============================

train_test = dataset.train_test_split(test_size=0.1)

datasets = DatasetDict({
    "train": train_test["train"],
    "validation": train_test["test"]
})


print(datasets)

DatasetDict({
    train: Dataset({
        features: ['ja', 'ar'],
        num_rows: 1712916
    })
    validation: Dataset({
        features: ['ja', 'ar'],
        num_rows: 190324
    })
})


In [133]:

print("First example:")
print(datasets["train"][0])
print(datasets["train"][1])
print(datasets["train"][2])
print(datasets["train"][3])
print("\nSample sizes:")
print({split: len(ds) for split, ds in datasets.items()})

First example:
{'ja': '- 私サイモンよ', 'ar': '-سأكون (سيمون ).'}
{'ja': '"こんにちは"とか "さよなら"...', 'ar': 'فقط مرحبا أو مع السلامة لا لا لا'}
{'ja': 'まさか それはあなたの仕事だ', 'ar': 'كلا. يمكنك جلب عامل نظافة.'}
{'ja': '"カルテルの人達も 苦しむべきです"', 'ar': 'هناك دائماً من عانى أكثر مني\u202c'}

Sample sizes:
{'train': 1712916, 'validation': 190324}


In [3]:

model_name = "Helsinki-NLP/opus-mt-ja-ar"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/852k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/904k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [134]:
# ────────────────────────────────────────────────
# 3. Preprocessing function
# ────────────────────────────────────────────────
# def preprocess_function(examples):

#     model_inputs = tokenizer(
#         examples["ja"],
#         max_length=96,
#         truncation=True,
#         padding="max_length"
#     )

#     labels = tokenizer(
#         text_target=examples["ar"], 
#         max_length=96, 
#         truncation=True, 
#         padding="max_length"
#     )

#     model_inputs["labels"] = labels["input_ids"]
#     return model_inputs
# Tokenization (adjust max_length as needed)
def preprocess(examples):
    inputs = examples["ja"]
    targets = examples["ar"]
    model_inputs = tokenizer(inputs, max_length=132, truncation=True, padding="max_length")
    
    labels = tokenizer(text_target=targets, max_length=132, truncation=True, padding="max_length")["input_ids"]
    model_inputs["labels"] = labels
    return model_inputs






In [145]:

tokenized_datasets = datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=datasets["train"].column_names
)


Tokenizing filtered data:   0%|          | 0/12465 [00:00<?, ? examples/s]

Tokenizing filtered data:   0%|          | 0/1661 [00:00<?, ? examples/s]

Tokenizing filtered data:   0%|          | 0/2493 [00:00<?, ? examples/s]

In [138]:
tokenized_datasets["train"]
print("\nSample sizes:")
print({split: len(ds) for split, ds in tokenized_datasets.items()})


Sample sizes:
{'train': 1712916, 'validation': 190324}


In [139]:
from transformers import Seq2SeqTrainingArguments 
# ────────────────────────────────────────────────
# 4. Training arguments (conservative settings)
# ────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
output_dir="./ja_ar_model",

    learning_rate=3e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    save_total_limit=2,

    predict_with_generate=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=500,

    fp16=True,

    dataloader_num_workers=2
)




In [140]:
# ==============================
# 9. Evaluation metrics
# ==============================

bleu = BLEU()
chrf = CHRF()

def compute_metrics(eval_preds):

    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    bleu_score = bleu.corpus_score(decoded_preds, [decoded_labels]).score
    chrf_score = chrf.corpus_score(decoded_preds, [decoded_labels]).score

    return {
        "bleu": bleu_score,
        "chrf": chrf_score
    }


In [146]:


# ==============================
# 8. Data collator
# ==============================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_datasets["train"],

    eval_dataset=tokenized_datasets["validation"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)




In [ ]:
import torch
import gc

# 1. Clear everything possible
torch.cuda.empty_cache()
gc.collect()

# 2. Enable PyTorch's expandable segments (reduces fragmentation)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 3. Print memory status again
!nvidia-smi
print("Allocated after clear:", torch.cuda.memory_allocated() / 1024**3, "GB")
print("Reserved after clear:", torch.cuda.memory_reserved() / 1024**3, "GB")

In [147]:
print("Train size:", len(tokenized_datasets["train"]))
print("Validation size:", len(tokenized_datasets["validation"]))

Train size: 12465
Validation size: 1661


In [148]:


# ────────────────────────────────────────────────
# 6. Start training
# ────────────────────────────────────────────────
trainer.train()


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
final_results = trainer.predict(tokenized_datasets["test"])
print("Final test results:")
print(final_results.metrics)

In [ ]:
trainer.save_model("/kaggle/working/final_model2")
tokenizer.save_pretrained("/kaggle/working/final_model2")

!zip -r final_model2.zip /kaggle/working/final_model2

from IPython.display import FileLink
FileLink(r'final_model2.zip')